# 03_bronze_to_silver

Transform Bronze Delta data into clean Silver tables with SCD Type 2 history.

### Silver responsibilities
- Read the four Bronze Delta tables.
- Extract FHIR identifiers and timestamps.
- Keep the resource payload for traceability.
- Deduplicate repeated versions of the same resource.
- Maintain `effective_from`, `effective_to`, and `is_current`.

**Bronze → Silver only. Gold analytical modeling is handled by `04_silver_to_gold`.**


In [1]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

StatementMeta(, 2b7f32c6-50a1-4b19-b994-ec318e9dc5c8, 3, Finished, Available, Finished, False)

In [ ]:
# Silver is the default lakehouse; Bronze is attached for reads.
BRONZE_TABLES_PATH = "abfss://c6f69522-a061-46b1-9621-a5dd84bc1504@onelake.dfs.fabric.microsoft.com/03445f95-cabf-4a70-8f09-ef7c8b1bcdd5/Tables"
SILVER_TABLES_PATH = "Tables"

RESOURCE_TYPES = [
    "patient",
    "encounter",
    "observation",
    "condition"
]

print(f"Bronze tables path: {BRONZE_TABLES_PATH}")
print(f"Silver tables path: {SILVER_TABLES_PATH}")


StatementMeta(, 2b7f32c6-50a1-4b19-b994-ec318e9dc5c8, 4, Finished, Available, Finished, False)

Bronze tables path: abfss://c6f69522-a061-46b1-9621-a5dd84bc1504@onelake.dfs.fabric.microsoft.com/03445f95-cabf-4a70-8f09-ef7c8b1bcdd5/Tables
Silver tables path: Tables


## 1. Read Bronze Delta Tables

Bronze contains structured FHIR resources and ingestion metadata for each resource type.


In [ ]:
def read_bronze_resource(resource_type: str):
    table_path = f"{BRONZE_TABLES_PATH}/bronze_{resource_type}"

    return (
        spark.read
        .format("delta")
        .load(table_path)
    )


patient_bronze_df = read_bronze_resource("patient")
patient_bronze_df.printSchema()

StatementMeta(, 2b7f32c6-50a1-4b19-b994-ec318e9dc5c8, 5, Finished, Available, Finished, False)

root
 |-- resource: struct (nullable = true)
 |    |-- _birthDate: struct (nullable = true)
 |    |    |-- extension: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- url: string (nullable = true)
 |    |    |    |    |-- valueDateTime: string (nullable = true)
 |    |-- active: boolean (nullable = true)
 |    |-- address: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- _line: array (nullable = true)
 |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |-- extension: array (nullable = true)
 |    |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |    |-- url: string (nullable = true)
 |    |    |    |    |    |    |    |-- valueString: string (nullable = true)
 |    |    |    |-- city: string (nullable = true)
 |    |    |    |-- country: string (nullable = true)
 |    |    |    |-- district: strin

## 2. Standardize the Silver Record

Expose the FHIR ID and `meta.lastUpdated` while keeping the full resource payload for traceability.


In [ ]:
def standardize_silver_resource(bronze_df, resource_type: str):
    id_column = f"{resource_type}_id"

    return (
        bronze_df
        .select(
            F.col("resource.id").alias(id_column),
            F.col("resource.meta.lastUpdated").alias("source_last_updated"),
            F.col("resource").alias("resource"),
            "resource_type",
            "ingestion_date",
            "source_file",
            "api_url_or_params",
            "extraction_timestamp",
            "data_saved_timestamp",
            "bronze_loaded_timestamp"
        )
        .filter(F.col(id_column).isNotNull())
        .withColumn("source_last_updated", F.to_timestamp("source_last_updated"))
        .withColumn("extraction_timestamp", F.to_timestamp("extraction_timestamp"))
        .withColumn("data_saved_timestamp", F.to_timestamp("data_saved_timestamp"))
        .withColumn(
            "record_hash",
            F.sha2(F.to_json(F.col("resource")), 256)
        )
        .withColumn("silver_loaded_timestamp", F.current_timestamp())
    )


StatementMeta(, 2b7f32c6-50a1-4b19-b994-ec318e9dc5c8, 6, Finished, Available, Finished, False)

## 3. Deduplicate Incoming Data

Keep the latest version of each FHIR resource ID in the incoming Bronze data.


In [ ]:
def deduplicate_latest(df, id_column: str):
    window = Window.partitionBy(id_column).orderBy(
        F.col("source_last_updated").desc_nulls_last(),
        F.col("bronze_loaded_timestamp").desc_nulls_last()
    )

    return (
        df.withColumn("row_num", F.row_number().over(window))
          .filter(F.col("row_num") == 1)
          .drop("row_num")
    )

StatementMeta(, 2b7f32c6-50a1-4b19-b994-ec318e9dc5c8, 7, Finished, Available, Finished, False)

## 4. SCD Type 2 Processing

Insert new resources, ignore unchanged records, and version changed records with effective dates.


In [ ]:
def apply_scd2(incoming_df, table_name: str, id_column: str):
    target_path = f"{SILVER_TABLES_PATH}/{table_name}"

    incoming = (
        incoming_df
        .withColumn(
            "effective_from",
            F.coalesce(
                F.col("source_last_updated"),
                F.col("extraction_timestamp"),
                F.col("silver_loaded_timestamp")
            )
        )
        .withColumn("effective_to", F.lit(None).cast("timestamp"))
        .withColumn("is_current", F.lit(True))
    )

    if not DeltaTable.isDeltaTable(spark, target_path):
        (
            incoming.write
            .format("delta")
            .mode("overwrite")
            .option("mergeSchema", "true")
            .save(target_path)
        )
        print(f"Created Silver table: {table_name}")
        return

    target = DeltaTable.forPath(spark, target_path)
    current = (
        target.toDF()
        .filter(F.col("is_current") == True)
    )

    comparison = (
        incoming.alias("i")
        .join(
            current.alias("c"),
            F.col(f"i.{id_column}") == F.col(f"c.{id_column}"),
            "left"
        )
        .select(
            F.col(f"i.{id_column}").alias("resource_id"),
            F.col("i.record_hash").alias("incoming_hash"),
            F.col("c.record_hash").alias("current_hash"),
            F.col("i.effective_from").alias("incoming_effective_from")
        )
    )

    changed_or_new = comparison.filter(
        F.col("current_hash").isNull()
        | (F.col("incoming_hash") != F.col("current_hash"))
    )

    changed_existing = changed_or_new.filter(
        F.col("current_hash").isNotNull()
    )

    if not changed_existing.rdd.isEmpty():
        target.alias("t").merge(
            changed_existing.alias("s"),
            f"t.{id_column} = s.resource_id AND t.is_current = true"
        ).whenMatchedUpdate(
            set={
                "effective_to": "s.incoming_effective_from",
                "is_current": "false"
            }
        ).execute()

    rows_to_insert = (
        incoming.alias("i")
        .join(
            changed_or_new.select("resource_id").alias("x"),
            F.col(f"i.{id_column}") == F.col("x.resource_id"),
            "inner"
        )
        .drop("resource_id")
    )

    if not rows_to_insert.rdd.isEmpty():
        rows_to_insert.write.format("delta").mode("append").option("mergeSchema", "true").save(target_path)

    print(f"Processed Silver table: {table_name}")


StatementMeta(, 2b7f32c6-50a1-4b19-b994-ec318e9dc5c8, 8, Finished, Available, Finished, False)

## 5. Process Patient

Process Patient with the reusable SCD Type 2 logic.


In [ ]:
patient_silver_df = standardize_silver_resource(
    patient_bronze_df,
    "patient"
)

patient_silver_df = deduplicate_latest(
    patient_silver_df,
    "patient_id"
)

apply_scd2(
    patient_silver_df,
    "silver_patient",
    "patient_id"
)

StatementMeta(, 2b7f32c6-50a1-4b19-b994-ec318e9dc5c8, 9, Finished, Available, Finished, False)

Created Silver table: silver_patient


## 6. Process Encounter, Observation, and Condition

Apply the same transformation to the remaining FHIR resources.


In [ ]:
for resource_type in ["encounter", "observation", "condition"]:
    print("=" * 70)
    print(f"Processing: {resource_type.upper()}")
    print("=" * 70)

    bronze_df = read_bronze_resource(resource_type)
    silver_df = standardize_silver_resource(bronze_df, resource_type)
    silver_df = deduplicate_latest(silver_df, f"{resource_type}_id")

    apply_scd2(
        silver_df,
        f"silver_{resource_type}",
        f"{resource_type}_id"
    )

StatementMeta(, 2b7f32c6-50a1-4b19-b994-ec318e9dc5c8, 10, Finished, Available, Finished, False)

Processing: ENCOUNTER


Created Silver table: silver_encounter
Processing: OBSERVATION


Created Silver table: silver_observation
Processing: CONDITION


Created Silver table: silver_condition


## 7. Silver Validation

Check that each Silver Delta table contains records.


In [ ]:
silver_validation = []

for resource_type in RESOURCE_TYPES:
    table_name = f"silver_{resource_type}"
    table_path = f"{SILVER_TABLES_PATH}/{table_name}"
    count = spark.read.format("delta").load(table_path).count()

    silver_validation.append(
        (table_name, count, "PASS" if count > 0 else "FAIL")
    )

validation_df = spark.createDataFrame(
    silver_validation,
    ["table_name", "record_count", "status"]
)

display(validation_df)

StatementMeta(, 2b7f32c6-50a1-4b19-b994-ec318e9dc5c8, 11, Finished, Available, Finished, True)

SynapseWidget(Synapse.DataFrame, 7bf0b40f-6186-475b-a4fa-4db750b7a29a)

## Silver Layer Complete

Expected tables:
- `silver_patient`
- `silver_encounter`
- `silver_observation`
- `silver_condition`

Next: load analytical Gold tables and reporting views.
